# Compare Benchmark Runs
**Source:** Based on the example [compare-hypothesis.ipynb](https://github.com/Kotlin/kotlinx-benchmark/blob/master/examples/compare-hypothesis.ipynb) of the `kotlinx-benchmark` repository.

Grabs the last two runs and compares their score.

In [92]:
%use serialization, dataframe, kandy

In [ ]:
@Serializable
public data class Benchmark(
    public val benchmark: String,
    public val mode: String,
    public val warmupIterations: Int,
    public val warmupTime: String,
    public val measurementIterations: Int,
    public val measurementTime: String,
    public val primaryMetric: PrimaryMetric,
    public val secondaryMetrics: Map<String, PrimaryMetric>,
    public val params: JsonObject? = null
)

@Serializable
public data class PrimaryMetric(
    public val score: Double,
    public val scoreError: Double,
    public val scoreConfidence: List<Double>,
    public val scorePercentiles: Map<String, Double>,
    public val scoreUnit: String,
    public val rawData: List<List<Double>>,
)

In [136]:
val nameLookupMap = mapOf(
    Pair("io.karpfen.features.BaseLineBenchmark.run", "Base-line"),
    Pair("io.karpfen.features.TogglePointOverheadBenchmark.run","Toggle-point overhead"),
)

In [133]:
import java.nio.file.Files
import java.nio.file.attribute.BasicFileAttributes
import kotlin.io.path.exists
import kotlin.io.path.forEachDirectoryEntry
import kotlin.io.path.isDirectory
import kotlin.io.path.listDirectoryEntries
import kotlin.io.path.readText

val runsDir = notebook.workingDir.resolve("../../../build/reports/benchmarks/main")
val outputFiles = runsDir.listDirectoryEntries()
    .filter { it.isDirectory() }
    .sortedByDescending { dir -> Files.readAttributes(dir, BasicFileAttributes::class.java).creationTime() }
    .subList(0, 2)
    .map { it.resolve("benchmark.json") }

In [132]:
val json = Json { ignoreUnknownKeys = true }
val newRun = json.decodeFromString<List<Benchmark>>(outputFiles.first().readText())
val oldRun = json.decodeFromString<List<Benchmark>>(outputFiles.last().readText())

In [184]:
import kotlinx.serialization.json.encodeToJsonElement

val oldDf = oldRun.toDataFrame {
    "benchmark" from { nameLookupMap[it.benchmark]?: it.benchmark }
    "score" from { it.primaryMetric.score }
}
val newDf = newRun.toDataFrame {
    "benchmark" from { nameLookupMap[it.benchmark]?: it.benchmark }
    "score" from { it.primaryMetric.score }
}
val combinedDf = oldDf.concat(newDf).sortByDesc("score")
combinedDf

benchmark,score
Toggle-point overhead,"4,058790"
Base-line,"3,977662"


In [185]:
val scores = combinedDf["score"].values().map { (it as Number).toDouble() }
val minScore = scores.minOrNull() ?: 0.0
val maxScore = scores.maxOrNull() ?: 0.0

val lowerBound = minScore * 0.99
val upperBound = maxScore * 1.01

val plot = combinedDf.plot {
    bars {
        x("benchmark")
        y("score") {
            scale = continuous(lowerBound..upperBound)
        }
    }
    coordinatesTransformation = CoordinatesTransformation.cartesianFlipped()
    layout {
        this.xAxisLabel = ""
        this.yAxisLabel = "ms/1000 ticks"
        style {
            global {
                title {
                    margin(10.0, -10.0)
                }
                text {
                    fontFamily = FontFamily.MONO
                }
            }
        }
        // Adjust the height of the Kandy plot based on the number of tests.
        size = 800 to ((50 * newDf.size().nrow) + 100)
    }
}
DISPLAY(HTML("<h4>Comparison</h4>"))
DISPLAY(plot)

Comparison

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="iTawRX" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 150.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("iTawRX");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"guides":{
"x":{
"title":""
},
"y":{
"title":"ms/1000 ticks"
}
},
"coord":{
"name":"flip",
"flip":true
},
"data":{
"score":[4.058789674946866,3.97766207836193],
"benchmark":["Toggle-point overhead","Base-line"]
},
"ggsize":{
"width":800.0,
"height":150.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[3.9378854575783104,4.099377571696335]
}],
"layers":[{
"mapping":{
"x":"benchmark",
"y":"score"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"theme":{
"text":{
"family":"mono",
"blank":false
},
"title":{
"margin":[10.0,-10.0,10.0,-10.0],
"blank":false
},
"axis_ontop":false,
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"benchmark"
},{
"type":"float",
"column":"score"
}]
},
"spec_id":"160"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 3.94 
 
 
 
 
 
 
 3.96 
 
 
 
 
 
 
 3.98 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 4.02 
 
 
 
 
 
 
 4.04 
 
 
 
 
 
 
 4.06 
 
 
 
 
 
 
 4.08 
 
 
 
 
 
 
 4.1 
 
 
 
 
 
 
 
 
 
 
 Toggle-point overhead 
 
 
 
 
 
 
 
 
 Base-line 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ms/1000 ticks